Getting 15 - minute discharge data from 2021-2025   **netcdf, zarr

In [ ]:
import requests
import pandas as pd
from tqdm.auto import tqdm
import os
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

In [ ]:
# downloading raw discharge data at 15 minute interval from USGS

# ---------------------------------------------------------------
# LOAD GAGES
# ---------------------------------------------------------------

gages_df = pd.read_csv("huc8_gages/huc8_events_and_gages.csv", dtype={"STAID": str})

gage_ids = (
    gages_df["STAID"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.replace(".0", "", regex=False)
    .str.zfill(8)
    .drop_duplicates()
    .tolist()
)

print(f"Found {len(gage_ids)} unique gages")
print(gage_ids[:10])

# ---------------------------------------------------------------
# SETTINGS
# ---------------------------------------------------------------

start_date = "2021-01-01"
end_date = "2025-12-31"
parameter_code = "00060"

# ---------------------------------------------------------------
# FUNCTION
# ---------------------------------------------------------------


def get_discharge(site):
    url = "https://waterservices.usgs.gov/nwis/iv/"

    params = {
        "format": "json",
        "sites": site,
        "parameterCd": parameter_code,
        "startDT": start_date,
        "endDT": end_date,
        "siteStatus": "all",
    }

    r = requests.get(url, params=params)
    r.raise_for_status()

    js = r.json()
    series = js["value"].get("timeSeries", [])

    if len(series) == 0:
        return pd.DataFrame()

    rows = []

    for ts in series:
        info = ts["sourceInfo"]

        site_no = info["siteCode"][0]["value"]
        site_name = info["siteName"]

        lat = info["geoLocation"]["geogLocation"]["latitude"]
        lon = info["geoLocation"]["geogLocation"]["longitude"]

        units = ts["variable"]["unit"]["unitCode"]

        values = ts["values"][0].get("value", [])

        for obs in values:
            val = obs.get("value")

            rows.append(
                {
                    "STAID": site_no,
                    "site_name": site_name,
                    "datetime": obs["dateTime"],
                    "discharge_cfs": float(val) if val not in [None, ""] else None,
                    "qualifiers": ",".join(obs.get("qualifiers", [])),
                    "units": units,
                    "latitude": lat,
                    "longitude": lon,
                }
            )

    return pd.DataFrame(rows)


# ---------------------------------------------------------------
# DOWNLOAD
# ---------------------------------------------------------------

all_data = []
failed = []
empty = []

for gage in tqdm(gage_ids):
    try:
        df = get_discharge(gage)

        if df.empty:
            empty.append(gage)
        else:
            all_data.append(df)

    except Exception as e:
        failed.append((gage, str(e)))
        print(f"{gage}: {e}")

# ---------------------------------------------------------------
# COMBINE SAFELY
# ---------------------------------------------------------------

print(f"Successful gages: {len(all_data)}")
print(f"Empty gages: {len(empty)}")
print(f"Failed gages: {len(failed)}")

if len(all_data) > 0:
    discharge = pd.concat(all_data, ignore_index=True)

    discharge["datetime"] = pd.to_datetime(discharge["datetime"])
    discharge = discharge.sort_values(["STAID", "datetime"])

    discharge.to_csv("huc8_gages/usgs_discharge_2021_2025.csv", index=False)

    print(discharge.head())

else:
    discharge = pd.DataFrame()
    print("No discharge data found.")
    print("First 20 empty gages:")
    print(empty[:20])
    print("First 20 failed gages:")
    print(failed[:20])

In [ ]:
print(f"{len(empty)} gages returned no discharge data.")
print(empty[:20])  # first 20

In [ ]:
# changing LST time to UTC

discharge = pd.read_csv(
    "huc8_gages/usgs_discharge_2021_2025.csv",
    dtype={"STAID": str},
)

# Parse mixed-offset Eastern timestamps and convert to UTC
discharge["datetime"] = pd.to_datetime(
    discharge["datetime"],
    errors="coerce",
    utc=True,
)

print(discharge["datetime"].isna().sum(), "bad datetime values")

discharge = discharge.dropna(subset=["datetime"])

# Remove timezone while keeping UTC clock time
discharge["datetime"] = discharge["datetime"].dt.tz_localize(None)

# ---------------------------------------------------------------
# Fixed study period in UTC: 2021-2025
# ---------------------------------------------------------------

study_start = pd.Timestamp("2021-01-01 00:00")
study_end = pd.Timestamp("2025-12-31 23:45")

discharge = discharge[
    (discharge["datetime"] >= study_start) & (discharge["datetime"] <= study_end)
].copy()

expected_records = len(pd.date_range(study_start, study_end, freq="15min"))

coverage = []

for gage, df in discharge.groupby("STAID"):
    df = df.sort_values("datetime").set_index("datetime")

    observed_records = df["discharge_cfs"].resample("15min").count().gt(0).sum()

    coverage.append(
        {
            "STAID": gage,
            "observed_records": observed_records,
            "expected_records": expected_records,
            "coverage": observed_records / expected_records,
            "coverage_percent": round(100 * observed_records / expected_records, 2),
        }
    )

coverage = pd.DataFrame(coverage).sort_values("coverage_percent", ascending=False)

coverage.head()

In [ ]:
discharge.to_csv(
    "huc8_gages/usgs_discharge_2021_2025_UTC.csv",
    index=False,
)

print("UTC discharge data saved to huc8_gages/usgs_discharge_2021_2025_UTC.csv")

In [ ]:
raw = pd.read_csv("huc8_gages/usgs_discharge_2021_2025_UTC.csv", nrows=5)

print(raw["datetime"])

In [ ]:
raw = pd.read_csv("huc8_gages/usgs_discharge_2021_2025_UTC.csv")

print(raw["datetime"].str[-6:].value_counts())

Resampling to 15 min intervals for all gages

In [ ]:
# resampling to 15 minute intervals

# ---------------------------------------------------------------
# RESAMPLE TO 15-MINUTE UTC GRID
# ---------------------------------------------------------------

discharge_15min = (
    discharge.set_index("datetime")
    .groupby("STAID")
    .resample("15min")[["discharge_cfs", "latitude", "longitude"]]
    .max()
    .reset_index()
)

# Add site names back
site_names = discharge[["STAID", "site_name"]].drop_duplicates()

discharge_15min = discharge_15min.merge(
    site_names,
    on="STAID",
    how="left",
)

# Reorder columns
discharge_15min = discharge_15min[
    [
        "STAID",
        "site_name",
        "datetime",
        "discharge_cfs",
        "latitude",
        "longitude",
    ]
]

discharge_15min = discharge_15min.sort_values(
    ["STAID", "datetime"],
)

print(discharge_15min.head())

In [ ]:
summary = (
    discharge_15min.assign(minute=discharge_15min["datetime"].dt.minute)
    .groupby("STAID")["minute"]
    .apply(lambda x: sorted(x.unique()))
)

print(summary)

In [ ]:
discharge_15min.to_csv(
    "huc8_gages/usgs_discharge_2021_2025_UTC_15min.csv",
    index=False,
)

print("Saved UTC 15-minute discharge data.")

In [ ]:
df = pd.read_csv("huc8_gages/usgs_discharge_2021_2025_UTC_15min.csv")

print(df.columns)

In [ ]:
qualifiers = discharge.groupby("STAID")["qualifiers"].unique().reset_index()

print(qualifiers.to_string(index=False))

In [ ]:
for gage, df in discharge.groupby("STAID"):
    print(f"\n{gage}")
    print(df["qualifiers"].value_counts())

In [ ]:
# plotting

# ---------------------------------------------------------------
# LOAD DATA
# ---------------------------------------------------------------

discharge = pd.read_csv(
    "huc8_gages/usgs_discharge_2021_2025_UTC_15min.csv",
    dtype={"STAID": str},
)

discharge["datetime"] = pd.to_datetime(
    discharge["datetime"],
    errors="coerce",
)

discharge = discharge.dropna(subset=["datetime"])

# Optional fixed study period
study_start = pd.Timestamp("2021-01-01 00:00")
study_end = pd.Timestamp("2025-12-31 23:45")

discharge = discharge[
    (discharge["datetime"] >= study_start) & (discharge["datetime"] <= study_end)
].copy()

# ---------------------------------------------------------------
# OUTPUT FOLDER
# ---------------------------------------------------------------

out_dir = "huc8_gages/discharge_plots"
os.makedirs(out_dir, exist_ok=True)

# ---------------------------------------------------------------
# PLOT ONE FIGURE PER GAGE
# ---------------------------------------------------------------

for gage, df in discharge.groupby("STAID"):
    df = df.sort_values("datetime")

    site_name = df["site_name"].iloc[0]

    fig, ax = plt.subplots(figsize=(14, 4.5))

    ax.plot(
        df["datetime"],
        df["discharge_cfs"],
        linewidth=0.6,
    )

    ax.set_title(
        f"USGS {gage} — {site_name}\nDischarge Time Series, 2021–2025",
        fontsize=13,
    )

    ax.set_xlabel("Date (UTC)")
    ax.set_ylabel("Discharge (cfs)")

    ax.grid(True, alpha=0.3)

    # Nice yearly ticks
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    ax.set_xlim(study_start, study_end)

    fig.tight_layout()

    out_path = os.path.join(out_dir, f"{gage}_discharge_2021_2025.png")
    fig.savefig(out_path, dpi=300, bbox_inches="tight")

    plt.close(fig)

print(f"Saved plots to: {out_dir}")

In [ ]:
# checking for gaps in the record

gap_summary = []

for gage, df in discharge_15min.groupby("STAID"):
    df = df.sort_values("datetime").copy()

    missing = df["discharge_cfs"].isna()

    groups = (missing != missing.shift()).cumsum()

    gaps = df[missing].groupby(groups[missing]).size()

    largest_gap = gaps.max() if len(gaps) else 0

    gap_summary.append(
        {
            "STAID": gage,
            "largest_gap_records": largest_gap,
            "largest_gap_days": round(largest_gap * 15 / 60 / 24, 2),
        }
    )

gap_summary = pd.DataFrame(gap_summary)

print(gap_summary.sort_values("largest_gap_days", ascending=False))

In [ ]:
# removing rows with nonvalid gages

empty = [
    '208726005',
    '208758850',
    '208675010',
    '208735012',
    '02088470',
    '208732885',
    '208521324',
    '02085220',
    '02086000',
    '208524090',
    '208524975',
    '208650112',
    '208773375',
]

# Load CSV, forcing STAID as string
combined = pd.read_csv(
    "huc8_gages/huc8_storms_plus_extra_gages_with_divides_filtered.csv",
    dtype={"STAID": str},
)

# Clean STAID formatting
combined["STAID_clean"] = (
    combined["STAID"].astype(str).str.replace(".0", "", regex=False).str.zfill(8)
)

empty_clean = (
    pd.Series(empty).astype(str).str.replace(".0", "", regex=False).str.zfill(8)
)

# Remove gages with no discharge data
combined = combined[~combined["STAID_clean"].isin(empty_clean)].copy()

# Drop helper column
combined = combined.drop(columns="STAID_clean")

# Save
combined.to_csv(
    "huc8_gages/huc8_events_and_gages_unique_with_divides_filtered.csv",
    index=False,
)

print(f"Saved {len(combined)} rows after removing no-data gages.")